# Dissertation figures — Synthetic Pelvic MRI Augmentation

Publication-quality figures generated **from the project's own result files**
(`metrics/master_metrics.csv`, `metrics/variance_study_summary.json`,
`*_dsc_summary.json`). Run the notebook top-to-bottom: every figure renders
inline, and the **final cell saves them all to `figures/` at 300 dpi**.

Because the charts read the result files live, re-running after a pending
experiment finishes will update the affected figures automatically.

> **Setup:** open Jupyter from the repository root so the relative paths
> resolve. If you run from elsewhere, edit `ROOT` in the first code cell.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- where things live (edit ROOT if you run from elsewhere) ----
ROOT = Path.cwd()
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

# ---- house style: clean, consistent, print-ready ----
plt.rcParams.update({
    "figure.dpi": 120,          # crisp inline preview
    "savefig.dpi": 150,         # print quality but keeps charts <= 2000px
    "savefig.bbox": "tight",
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
    "legend.frameon": False,
})

# consistent palette across every figure
C = {
    "baseline": "#404040",   # real-only reference line
    "concat":   "#C44E52",   # concat conditioning (the failure mode)
    "concat_l": "#E39399",   # lighter concat
    "spade":    "#4C72B0",   # SPADE conditioning
    "spade_l":  "#9BB4D6",   # lighter SPADE
    "phase2":   "#8172B3",   # cross-domain Phase 2
    "accent":   "#DD8452",   # highlight (e.g. n=3 reading)
}

BASELINE = 0.290   # RAovSeg real-only ovary DSC (Liang et al., 2025) — verified reproduction

FIGS = {}          # every figure is collected here for the save-all cell
print("ROOT      :", ROOT)
print("FIG_DIR   :", FIG_DIR)

## 1. Load the result files

All downstream DSC numbers come from the per-seed summary JSONs; all generator
quality metrics come from `master_metrics.csv`. Nothing here is hand-typed.

In [ ]:
def load_json(name):
    p = ROOT / name
    if not p.exists():
        p = ROOT / "metrics" / Path(name).name
    return json.loads(p.read_text())

def seed_means(summary):
    "Mean DSC per completed seed; ignores not_started / failed seeds."
    return np.array([s["dsc_mean"] for s in summary["seeds"] if s.get("status") == "ok"])

variance   = load_json("metrics/variance_study_summary.json")
spade_seeds = seed_means(variance)                       # SPADE v3, n=8 seeds
exp2   = seed_means(load_json("exp2_dsc_summary.json"))       # Phase 2, lambda=0.01
exp2C  = seed_means(load_json("exp2_pathC_dsc_summary.json")) # Phase 2, Path C
lam05  = seed_means(load_json("lam05_dsc_summary.json"))      # Phase 2, lambda=0.05

quality = pd.read_csv(ROOT / "metrics" / "master_metrics.csv")

# The five data charts live in a reusable module so every figure is
# reproducible from a script, not just from these notebook cells.
import sys
sys.path.insert(0, str(ROOT))
from IPython.display import Image, display
from src.RaovSeg_recreation import make_data_charts as DC
dc_data = DC._load(ROOT)   # loaded once, reused by every chart cell below

print(f"SPADE v3 seeds (n={spade_seeds.size}): "
      f"mean={spade_seeds.mean():.3f}  sd={spade_seeds.std():.3f}")
print(f"Phase 2 exp2   (n={exp2.size}):  mean={exp2.mean():.3f}")
print(f"Phase 2 pathC  (n={exp2C.size}): mean={exp2C.mean():.3f}")
print(f"Phase 2 lam05  (n={lam05.size}): mean={lam05.mean():.3f}")
quality[["variant","fid","hist_kl","lpips_nn_mean",
         "CLR_uterus","CLR_ov_L","CLR_em"]].round(3)

## 2. Downstream augmentation trajectory (v1 → v2 → v3)

The headline result: across every preprocessing-fix level, neither conditioning
scheme reaches the real-only baseline. SPADE climbs but plateaus; concat collapses.

*v1/v2 values are the canonical figures from the dissertation drafts; the v3
SPADE point is computed live from `variance_study_summary.json`.* Each chart
below is drawn by `make_data_charts`, which saves the PNG straight into
`figures/` — so the figure is reproducible by running this cell *or* the script.

In [ ]:
DC.fig_downstream_trajectory(dc_data, FIG_DIR)
display(Image(filename=str(FIG_DIR / "fig_downstream_trajectory.png")))

## 3. Phase 2 cross-domain collapse

Moving the generator cross-domain (D1 T2 -> D2 T2FS) does not recover the gap —
it makes things dramatically worse. `exp2` (lambda=0.01) collapses to near-zero.

In [ ]:
DC.fig_phase2_collapse(dc_data, FIG_DIR)
display(Image(filename=str(FIG_DIR / "fig_phase2_collapse.png")))

## 4. Seed variance study (n = 8)

A methodological point: the optimistic reading came from only the first 3 seeds
(mean 0.218). Five more seeds drag the mean down to 0.178, well below baseline —
so small seed counts overstate augmentation benefit. Fully data-driven from
`variance_study_summary.json`.

In [ ]:
DC.fig_variance_study(dc_data, FIG_DIR)
display(Image(filename=str(FIG_DIR / "fig_variance_study.png")))

## 5. Generator quality metrics across the 2x2

No single variant wins on distributional quality: concat variants win FID and
hist_KL, a SPADE variant wins LPIPS. These standard metrics therefore fail to
predict downstream utility — motivating the task-relevant CLR metric (next).
Read live from `master_metrics.csv` (concat = red, SPADE = blue).

In [ ]:
DC.fig_quality_metrics(dc_data, FIG_DIR)
display(Image(filename=str(FIG_DIR / "fig_quality_metrics.png")))

## 6. Per-organ localisation (CLR)

The task-relevant metric. Counterfactual Localisation Ratio measures whether
removing an organ's label channel changes the image *locally* at that organ.
SPADE variants localise (CLR ~0.3-0.5); concat variants do not (CLR ~0.03-0.08) —
the mechanism behind "concat is architecturally locked out".

In [ ]:
DC.fig_clr_localisation(dc_data, FIG_DIR)
display(Image(filename=str(FIG_DIR / "fig_clr_localisation.png")))

## 7. Orientation diagrams + summary tables (no data needed)

These are drawn by the two helper scripts in
`src/RaovSeg_recreation/`. Running the cell regenerates them into
`figures/` and shows them inline.

In [ ]:
import sys
from IPython.display import Image, display
sys.path.insert(0, str(ROOT))

from src.RaovSeg_recreation import make_diagrams_and_tables as D
from src.RaovSeg_recreation import make_architecture_figures as A

# diagrams + tables (pure matplotlib, no data dependency)
D.fig_pipeline(FIG_DIR / "fig_pipeline.png")
D.fig_ablation_2x2(FIG_DIR / "fig_ablation_2x2.png")
D.fig_conditioning(FIG_DIR / "fig_conditioning_schematic.png")
D.fig_per_subject_dsc(FIG_DIR)
D.table_dataset(FIG_DIR)
D.table_results(FIG_DIR)
A.fig_ddpm(FIG_DIR / "fig_ddpm_process.png")
A.fig_architecture(FIG_DIR / "fig_architecture.png")
# clipart label fallback; overwritten by the real-anatomy overlay in Section 8
# whenever SimpleITK + the data are available.
A.fig_label_channels(FIG_DIR / "fig_label_channels.png")

# CLR counterfactual panel — cropped from cached explain reports (guarded)
try:
    from src.RaovSeg_recreation import make_clr_counterfactual as CF
    CF.make(ROOT, FIG_DIR)
except Exception as e:
    print("CLR counterfactual skipped:", e)

# results figures: heat-styled metric matrix, DSC forest plot, exp2 collapse strip
try:
    from src.RaovSeg_recreation import make_result_figures as RF
    RF.fig_quality_heatmatrix(ROOT, FIG_DIR)
    RF.fig_dsc_forest(FIG_DIR)
    RF.fig_exp2_collapse(ROOT, FIG_DIR)
except Exception as e:
    print("result figures skipped:", e)

# methodology schematics: RAovSeg arch, reproduction heat-table, ISCS illustration
try:
    from src.RaovSeg_recreation import make_methodology_figures as MF
    MF.fig_raovseg_arch(FIG_DIR)
    MF.fig_repro_heat(FIG_DIR)
    MF.fig_iscs(FIG_DIR)
except Exception as e:
    print("methodology figures skipped:", e)

for name in ["fig_pipeline", "fig_ablation_2x2", "fig_conditioning_schematic",
             "fig_clr_counterfactual", "fig_quality_heatmatrix", "fig_dsc_forest",
             "fig_exp2_collapse", "fig_per_subject_dsc", "fig_ddpm_process",
             "fig_architecture", "fig_raovseg_arch", "fig_repro_heat", "fig_iscs",
             "table_dataset_summary", "table_d2_filtering", "table_master_results"]:
    p = FIG_DIR / f"{name}.png"
    if p.exists():
        display(Image(filename=str(p)))

## 8. Mechanism figures (needs SimpleITK + local synth volumes)

Real-vs-synth overlay + intensity histograms + the real-anatomy
six-channel label figure (Fig 3.2), rendered from the actual NIfTI
volumes. This step is skipped gracefully if SimpleITK isn't installed
or the synth-volume folders aren't present. It includes Phase 1
(`1c_spade`) and, when pulled, Phase 2 (`exp2`).

In [ ]:
import subprocess
mech_ok = False
try:
    import SimpleITK  # noqa: F401
    synth_dirs = ["1c SPADE (Phase 1)=exp1c_spade_samples"]
    if (ROOT / "exp2_samples_volumes").exists():
        synth_dirs.append("exp2 (Phase 2)=exp2_samples_volumes")
    cmd = [sys.executable, "-m", "src.RaovSeg_recreation.mechanism_figures",
           "--real-dir", "UT-EndoMRI/D2_TCPW",
           "--real-subjects", "D2-016", "D2-017", "D2-024",
           "--synth-dirs", *synth_dirs,
           "--synth-subjects", "D2-900", "D2-901", "D2-902",
           "--out-dir", "figures"]
    r = subprocess.run(cmd, cwd=str(ROOT), capture_output=True, text=True, timeout=900)
    print(r.stdout[-1500:])
    mech_ok = (r.returncode == 0)
    if not mech_ok:
        print("mechanism step exit", r.returncode, "\n", r.stderr[-800:])
except Exception as e:
    print("Skipping mechanism figures:", type(e).__name__, e)

if mech_ok:
    for name in ["fig_mech_overlay", "fig_mech_ovary_hist", "fig_mech_body_hist",
                 "fig_label_channels", "fig_pipeline_stages", "fig_failure_modes",
                 "fig_preprocess_progression"]:
        p = FIG_DIR / f"{name}.png"
        if p.exists():
            display(Image(filename=str(p)))

## 9. Cap every figure to ≤ 2000 px

Keeps files small and lets the figures attach to image-limited chat
interfaces without the "exceeds 2000px" error.

In [ ]:
try:
    from src.RaovSeg_recreation import cap_figures
    cap_figures.cap(FIG_DIR, 2000)
except Exception as e:
    print("cap step skipped:", e)

## 10. Index — everything now in `figures/`

In [ ]:
for p in sorted(FIG_DIR.glob("*.png")) + sorted(FIG_DIR.glob("*.csv")):
    print(p.name)